# Random Forest & the CMS classification of CRC

## Introduction

### Background
* The consensus molecular subtype (CMS) classification divides colorectal cancer into 4 subtypes (CMS1-4) based on gene expression, using a Random Forest classifier.
* Random Forest (RF) is a robust method for classification, especially with high-dimensional data like gene expression.

### Task overview
* In this exercise you will:
    1. Load and pre-process a dataset from colorectal cancer patients, labeled for CMS.
    2. Train a RF model.
    3. Evaluate the model.
    4. Optional: Plot top features

## Setup

### Import libraries
Run the cell bellow to import the required libraries.  
If any are missing, install them using `!pip install <library>`

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

### Load the dataset
TCGA data: TCGA-COAD (colon adenocarcinoma) and TCGA-READ (rectum adenocarcinoma).

The dataset contains gene expression values (rows: samples, columns: genes) and an additional column `CMS` of the CMS labels.  
This dataset contains a small selected subset of the genes, for simplification.  

In [ ]:
# Load data
data_gene_exp = pd.read_csv("/Users/luanadoaido/ZHAW/HS25/Track1/Mini Project/Track/STAD/TCGA-STAD_gene_expression_cpm.csv", index_col=0)
print("Data shape:", data_gene_exp.shape)

#431 Features und 60616 Observations

Data shape: (431, 60616)


In [5]:
data_gene_exp.head()

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288661,ENSG00000288662,ENSG00000288663,ENSG00000288665,ENSG00000288667,ENSG00000288669,ENSG00000288670,ENSG00000288671,ENSG00000288674,ENSG00000288675
submitter_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3M-AB46,41.435372,0.099263,58.437771,15.896322,10.819709,2.041990,11.656357,105.800585,48.724140,35.919164,...,0.0,0.085083,0.581400,0.0,0.127624,0.000000,4.693740,0.0,0.042541,0.467956
TCGA-3M-AB47,40.888469,0.027332,26.689566,9.989796,3.184162,6.955960,135.169736,32.019279,39.152896,18.845321,...,0.0,0.054664,0.560303,0.0,0.081996,0.013666,3.648804,0.0,0.095662,0.601301
TCGA-B7-5816,19.181703,0.575894,14.234920,6.807363,4.976315,18.650108,79.753946,70.288610,20.126760,39.456127,...,0.0,0.014767,0.177198,0.0,0.014767,0.000000,1.712916,0.0,0.044300,0.664493
TCGA-B7-5818,28.038900,0.000000,37.604185,9.565285,4.677530,20.575875,17.054588,77.179239,47.195749,33.347108,...,0.0,0.078835,0.446730,0.0,0.000000,0.000000,1.051130,0.0,0.105113,0.656956
TCGA-B7-A5TI,24.049394,0.000000,39.664949,14.982297,7.483953,5.857632,51.840764,50.531072,43.464494,24.725828,...,0.0,0.000000,0.388590,0.0,0.000000,0.000000,4.245704,0.0,0.043177,0.532512


## Data pre-processing

### Separate features and labels
We want to separate the features and the labels into `X` and `y`.

In [ ]:
# Separate features and labels
X = data.drop("CMS", axis=1)    # Feature columns: gene expression data
y = data["CMS"]                 # Target column: CMS labels

### Split the data into train/test sets
Now we split the data into train/test sets.  
The `test_size` is set to `0.3`, but you can change it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(X_train.shape)
print(X_test.shape)

## Train the Random Forest model

In [ ]:
# Train the model
rfc = RandomForestClassifier(n_estimators=101, bootstrap=True, max_depth=70, max_features=20, criterion='gini', random_state=42)
rfc.fit(X_train, y_train)

## Evaluate model performance

### Predeict y for the test set

In [ ]:
# Predict the CMS labels of the test set
y_pred = rfc.predict(X_test)

In [ ]:
# confusion_matrix
# Rows: true labels
# Columns: predicted labels
confusion_matrix(y_test, y_pred,labels= rfc.classes_)

### Calculate metrics
*Here you will need to adjust the code.*  

*What is the difference between `accuracy_score` and `balanced_accuracy_score`?*

In [ ]:
# Calculate scores
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred, average='weighted'))

print("Recall:", recall_score(y_test, y_pred, average='weighted'))

print("F1-score:", f1_score(y_test, y_pred, average='weighted'))

In [ ]:
print("F1-score:", f1_score(y_test, y_pred, average=None))

### Predict probabilities
We can access the probabilities of predicting each class for each sample in the test set.  

*What do these probabilities represent?*

In [ ]:
y_pred_proba = pd.DataFrame(rfc.predict_proba(X_test),
                            index=X_test.index, columns=rfc.classes_)

In [ ]:
y_pred_proba.head()

## Feature importance (optional)
We can visualise the top most important genes.

In [ ]:
importances = pd.DataFrame({
    'feature': rfc.feature_names_in_,
    'importance': rfc.feature_importances_
})
top_importances = importances.nlargest(n=10, columns='importance')

In [ ]:
# Plot
plt.figure(figsize=(10,6))
plt.title("Top 10 important genes")
sns.barplot(data=top_importances, x='importance', y='feature')